# 04_promotion_split_260513

프로모션 split 계약, 기초 group distribution, target comparison, duration anomaly comparison을 고정하는 감사 notebook입니다.

이 단계에서는 modeling, prediction, SHAP, Optuna, feature engineering, row exclusion, duplicate removal, causal claim을 수행하지 않습니다.

In [1]:
from pathlib import Path
from datetime import datetime
from zipfile import ZipFile

import pandas as pd
import numpy as np

PARK_ROOT = Path(r"C:\\Code\\ott-churn-prediction\\park.ingyeom").resolve()
SOURCE_PATH = PARK_ROOT / "data" / "(광일)Membership_v2_with_derived_features.csv"
PREV_01_DIR = PARK_ROOT / "reports" / "audits" / "01_data_contract_260513"
PREV_02_DIR = PARK_ROOT / "reports" / "audits" / "02_target_score_orientation_260513"
PREV_03_DIR = PARK_ROOT / "reports" / "audits" / "03_observation_window_policy_260513"
NOTEBOOK_PATH = PARK_ROOT / "notebook" / "04_promotion_split_260513" / "04_promotion_split_260513.ipynb"
BASE_OUTPUT_DIR = PARK_ROOT / "reports" / "audits" / "04_promotion_split_260513"
NOTE_PATH = PARK_ROOT / "note.md"
ZIP_DIR = PARK_ROOT / "zip"
ZIP_PATH = ZIP_DIR / "04_promotion_split_260513_review_package.zip"

def is_inside(child: Path, parent: Path) -> bool:
    try:
        child.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False

BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if any(BASE_OUTPUT_DIR.iterdir()):
    OUTPUT_DIR = BASE_OUTPUT_DIR / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
else:
    OUTPUT_DIR = BASE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

for p in [SOURCE_PATH, PREV_01_DIR, PREV_02_DIR, PREV_03_DIR, NOTEBOOK_PATH, OUTPUT_DIR, NOTE_PATH, ZIP_PATH]:
    assert is_inside(p, PARK_ROOT), f"Path outside park.ingyeom: {p}"

source_exists = SOURCE_PATH.exists()
previous_01_folder_exists = PREV_01_DIR.exists()
previous_02_folder_exists = PREV_02_DIR.exists()
previous_03_folder_exists = PREV_03_DIR.exists()
source_stat_before = SOURCE_PATH.stat() if source_exists else None
note_stat_before = NOTE_PATH.stat() if NOTE_PATH.exists() else None
written_files = []
warnings = []

print("SOURCE_PATH:", SOURCE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("ZIP_PATH:", ZIP_PATH)

SOURCE_PATH: C:\Code\ott-churn-prediction\park.ingyeom\data\(광일)Membership_v2_with_derived_features.csv
OUTPUT_DIR: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513
ZIP_PATH: C:\Code\ott-churn-prediction\park.ingyeom\zip\04_promotion_split_260513_review_package.zip


In [2]:
def save_csv(frame: pd.DataFrame, filename: str):
    path = OUTPUT_DIR / filename
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output: {path}")
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    written_files.append(path)
    print("saved:", path)
    return path

def read_prev_csv(path: Path, label: str):
    if path.exists():
        try:
            return pd.read_csv(path), "FOUND"
        except Exception as exc:
            warnings.append(f"previous output read failed: {label}: {exc}")
            return None, "WARNING_READ_FAILED"
    warnings.append(f"previous output missing: {label}")
    return None, "WARNING_MISSING"

prev_sources = {
    "01_promotion_target_2x2.csv": PREV_01_DIR / "01_promotion_target_2x2.csv",
    "01_duration_anomaly_audit.csv": PREV_01_DIR / "01_duration_anomaly_audit.csv",
    "02_target_contract.csv": PREV_02_DIR / "02_target_contract.csv",
    "02_score_orientation_policy.csv": PREV_02_DIR / "02_score_orientation_policy.csv",
    "03_observation_window_policy.csv": PREV_03_DIR / "03_observation_window_policy.csv",
    "03_duration_anomaly_by_group.csv": PREV_03_DIR / "03_duration_anomaly_by_group.csv",
    "03_final_checks.csv": PREV_03_DIR / "03_final_checks.csv",
}
prev_status_rows = []
for label, path in prev_sources.items():
    _, status = read_prev_csv(path, label)
    prev_status_rows.append({"item": f"previous_output::{label}", "actual_value": status, "previous_value": "", "match_previous": "", "status": status, "note": str(path)})

df = pd.read_csv(SOURCE_PATH)
row_count = int(len(df))
column_count = int(df.shape[1])
total_missing_count = int(df.isna().sum().sum())
unique_user_key_count = int(df["USER_KEY"].nunique(dropna=True)) if "USER_KEY" in df.columns else np.nan
duplicated_user_key_extra_rows = int(row_count - unique_user_key_count) if "USER_KEY" in df.columns else np.nan
duplicated_full_row_count = int(df.duplicated().sum())

promotion_distribution = df["is_promotion"].value_counts(dropna=False).rename_axis("is_promotion").reset_index(name="count") if "is_promotion" in df.columns else pd.DataFrame()
if not promotion_distribution.empty:
    promotion_distribution["rate"] = promotion_distribution["count"] / row_count
target_distribution = df["is_repurchase"].value_counts(dropna=False).rename_axis("is_repurchase").reset_index(name="count") if "is_repurchase" in df.columns else pd.DataFrame()
if not target_distribution.empty:
    target_distribution["rate"] = target_distribution["count"] / row_count

if {"is_promotion", "is_repurchase"}.issubset(df.columns):
    promotion_target_2x2 = df.groupby(["is_promotion", "is_repurchase"], dropna=False).size().reset_index(name="count")
    promotion_target_2x2["row_total_by_is_promotion"] = promotion_target_2x2.groupby("is_promotion", dropna=False)["count"].transform("sum")
    promotion_target_2x2["row_percentage_within_is_promotion"] = promotion_target_2x2["count"] / promotion_target_2x2["row_total_by_is_promotion"]
    promotion_target_2x2["overall_percentage"] = promotion_target_2x2["count"] / row_count
else:
    promotion_target_2x2 = pd.DataFrame()

parsed_reg = pd.to_datetime(df["reg_date"], errors="coerce") if "reg_date" in df.columns else pd.Series([pd.NaT] * row_count)
parsed_end = pd.to_datetime(df["end_date"], errors="coerce") if "end_date" in df.columns else pd.Series([pd.NaT] * row_count)
duration_days = (parsed_end - parsed_reg).dt.days
duration_lt_21 = duration_days < 21
duration_eq_0 = duration_days == 0
duration_1_20 = (duration_days >= 1) & (duration_days <= 20)
duration_21_30 = (duration_days >= 21) & (duration_days <= 30)
duration_gte_21 = duration_days >= 21

def mask_count(mask):
    return int(mask.fillna(False).sum())

input_consistency = pd.DataFrame([
    {"item": "source_file_exists", "actual_value": bool(SOURCE_PATH.exists()), "previous_value": "", "match_previous": "", "status": "PASS" if SOURCE_PATH.exists() else "FAIL", "note": str(SOURCE_PATH)},
    {"item": "previous_01_folder_exists", "actual_value": bool(PREV_01_DIR.exists()), "previous_value": "", "match_previous": "", "status": "PASS" if PREV_01_DIR.exists() else "WARNING", "note": str(PREV_01_DIR)},
    {"item": "previous_02_folder_exists", "actual_value": bool(PREV_02_DIR.exists()), "previous_value": "", "match_previous": "", "status": "PASS" if PREV_02_DIR.exists() else "WARNING", "note": str(PREV_02_DIR)},
    {"item": "previous_03_folder_exists", "actual_value": bool(PREV_03_DIR.exists()), "previous_value": "", "match_previous": "", "status": "PASS" if PREV_03_DIR.exists() else "WARNING", "note": str(PREV_03_DIR)},
    {"item": "row_count", "actual_value": row_count, "previous_value": "", "match_previous": "", "status": "RECORDED", "note": "recomputed from source CSV"},
    {"item": "column_count", "actual_value": column_count, "previous_value": "", "match_previous": "", "status": "RECORDED", "note": "recomputed from source CSV"},
    {"item": "total_missing_count", "actual_value": total_missing_count, "previous_value": "", "match_previous": "", "status": "RECORDED", "note": "recomputed from source CSV"},
    {"item": "unique_USER_KEY_count", "actual_value": unique_user_key_count, "previous_value": "", "match_previous": "", "status": "RECORDED", "note": "recomputed from source CSV"},
    {"item": "duplicated_USER_KEY_extra_rows", "actual_value": duplicated_user_key_extra_rows, "previous_value": "", "match_previous": "", "status": "RECORDED", "note": "row_count - unique USER_KEY count"},
    {"item": "duplicated_full_row_count", "actual_value": duplicated_full_row_count, "previous_value": "", "match_previous": "", "status": "RECORDED", "note": "counted only; not removed"},
    {"item": "duration_lt_21_count", "actual_value": mask_count(duration_lt_21), "previous_value": "", "match_previous": "", "status": "RECORDED", "note": "computed from end_date - reg_date"},
    {"item": "duration_eq_0_count", "actual_value": mask_count(duration_eq_0), "previous_value": "", "match_previous": "", "status": "RECORDED", "note": "computed from end_date - reg_date"},
    {"item": "duration_21_30_count", "actual_value": mask_count(duration_21_30), "previous_value": "", "match_previous": "", "status": "RECORDED", "note": "computed from end_date - reg_date"},
    {"item": "duration_gte_21_count", "actual_value": mask_count(duration_gte_21), "previous_value": "", "match_previous": "", "status": "RECORDED", "note": "computed from end_date - reg_date"},
] + prev_status_rows)

display(input_consistency)
display(promotion_distribution)
display(target_distribution)
display(promotion_target_2x2)
save_csv(input_consistency, "04_input_consistency_check.csv")

,item,actual_value,previous_value,match_previous,status,note
0,source_file_exists,True,,,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...
1,previous_01_folder_exists,True,,,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
2,previous_02_folder_exists,True,,,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
3,previous_03_folder_exists,True,,,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
4,row_count,23343,,,RECORDED,recomputed from source CSV
5,column_count,91,,,RECORDED,recomputed from source CSV
6,total_missing_count,0,,,RECORDED,recomputed from source CSV
7,unique_USER_KEY_count,23134,,,RECORDED,recomputed from source CSV
8,duplicated_USER_KEY_extra_rows,209,,,RECORDED,row_count - unique USER_KEY count
9,duplicated_full_row_count,48,,,RECORDED,counted only; not removed


,is_promotion,count,rate
0,1,11955,0.512145
1,0,11388,0.487855


,is_repurchase,count,rate
0,1,16702,0.715504
1,0,6641,0.284496


,is_promotion,is_repurchase,count,row_total_by_is_promotion,row_percentage_within_is_promotion,overall_percentage
0,0,0,2746,11388,0.241131,0.117637
1,0,1,8642,11388,0.758869,0.370218
2,1,0,3895,11955,0.325805,0.166859
3,1,1,8060,11955,0.674195,0.345286


saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_input_consistency_check.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/04_promotion_split_260513/04_input_consistency_check.csv')

In [3]:
promotion_split_contract = pd.DataFrame([
    {"contract_item": "split_column", "value": "is_promotion", "interpretation": "top-level analysis split variable", "allowed_in_overall_model_candidate": "yes/review", "allowed_in_promotion_only_model": "no", "allowed_in_nonpromotion_only_model": "no", "reason": "project is promotion-centric; within a split the value is constant"},
    {"contract_item": "non_promotion_value", "value": 0, "interpretation": "non-promotion group", "allowed_in_overall_model_candidate": "yes/review", "allowed_in_promotion_only_model": "no", "allowed_in_nonpromotion_only_model": "no", "reason": "split indicator, not groupwise feature"},
    {"contract_item": "promotion_value", "value": 1, "interpretation": "100원딜 / promotion group", "allowed_in_overall_model_candidate": "yes/review", "allowed_in_promotion_only_model": "no", "allowed_in_nonpromotion_only_model": "no", "reason": "split indicator, not groupwise feature"},
])

def dup_full_rows_by_group(group_value):
    subset = df[df["is_promotion"] == group_value] if "is_promotion" in df.columns else pd.DataFrame()
    return int(subset.duplicated().sum()) if not subset.empty else 0

if "is_promotion" in df.columns:
    rows = []
    for val, sub in df.groupby("is_promotion", dropna=False):
        n = int(len(sub))
        uniq = int(sub["USER_KEY"].nunique(dropna=True)) if "USER_KEY" in sub.columns else np.nan
        rows.append({
            "is_promotion": val,
            "n": n,
            "rate": n / row_count,
            "unique_USER_KEY_count": uniq,
            "duplicated_USER_KEY_extra_rows": int(n - uniq) if not pd.isna(uniq) else np.nan,
            "duplicated_full_rows": int(sub.duplicated().sum()),
        })
    promotion_distribution_detail = pd.DataFrame(rows)
    promotion_distribution_detail.loc[len(promotion_distribution_detail)] = {
        "is_promotion": "TOTAL_CHECK",
        "n": int(promotion_distribution_detail["n"].sum()),
        "rate": float(promotion_distribution_detail["rate"].sum()),
        "unique_USER_KEY_count": unique_user_key_count,
        "duplicated_USER_KEY_extra_rows": duplicated_user_key_extra_rows,
        "duplicated_full_rows": duplicated_full_row_count,
    }
else:
    promotion_distribution_detail = pd.DataFrame()

if {"is_promotion", "is_repurchase"}.issubset(df.columns):
    rate_rows = []
    for val, sub in df.groupby("is_promotion", dropna=False):
        n = int(len(sub))
        repurchase_rate = float((sub["is_repurchase"] == 1).mean())
        nonrepurchase_rate = float((sub["is_repurchase"] == 0).mean())
        rate_rows.append({"is_promotion": val, "n": n, "repurchase_rate": repurchase_rate, "non_repurchase_rate": nonrepurchase_rate})
    promotion_repurchase_rate_comparison = pd.DataFrame(rate_rows).sort_values("is_promotion")
    try:
        nonpromo_rate = float(promotion_repurchase_rate_comparison.loc[promotion_repurchase_rate_comparison["is_promotion"] == 0, "repurchase_rate"].iloc[0])
        promo_rate = float(promotion_repurchase_rate_comparison.loc[promotion_repurchase_rate_comparison["is_promotion"] == 1, "repurchase_rate"].iloc[0])
        absolute_diff = promo_rate - nonpromo_rate
        relative_diff = absolute_diff / nonpromo_rate if nonpromo_rate else np.nan
    except Exception:
        absolute_diff = np.nan
        relative_diff = np.nan
    promotion_repurchase_rate_comparison["promotion_minus_nonpromotion_repurchase_rate_diff"] = absolute_diff
    promotion_repurchase_rate_comparison["relative_difference_vs_nonpromotion_descriptive_only"] = relative_diff
    promotion_repurchase_rate_comparison["interpretation_limit"] = "descriptive comparison only; no causal inference"
else:
    promotion_repurchase_rate_comparison = pd.DataFrame()

save_csv(promotion_split_contract, "04_promotion_split_contract.csv")
save_csv(promotion_distribution_detail, "04_promotion_distribution.csv")
save_csv(promotion_target_2x2, "04_promotion_target_2x2.csv")
save_csv(promotion_repurchase_rate_comparison, "04_promotion_repurchase_rate_comparison.csv")

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_promotion_split_contract.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_promotion_distribution.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_promotion_target_2x2.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_promotion_repurchase_rate_comparison.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/04_promotion_split_260513/04_promotion_repurchase_rate_comparison.csv')

In [4]:
masks = {
    "duration_lt_21": duration_lt_21,
    "duration_eq_0": duration_eq_0,
    "duration_1_20": duration_1_20,
    "duration_21_30": duration_21_30,
    "duration_gte_21": duration_gte_21,
}

def duration_by_groups(group_cols):
    rows = []
    if not set(group_cols).issubset(df.columns):
        return pd.DataFrame(rows)
    for group_values, idx in df.groupby(group_cols, dropna=False).groups.items():
        if not isinstance(group_values, tuple):
            group_values = (group_values,)
        group_label = "|".join([f"{c}={v}" for c, v in zip(group_cols, group_values)])
        group_index = list(idx)
        denom = len(group_index)
        for metric, mask in masks.items():
            cnt = int(mask.loc[group_index].fillna(False).sum())
            rows.append({
                "grouping": "|".join(group_cols),
                "group_value": group_label,
                "metric": metric,
                "row_count": denom,
                "count": cnt,
                "rate_within_group": cnt / denom if denom else np.nan,
            })
    return pd.DataFrame(rows)

promotion_duration_anomaly = duration_by_groups(["is_promotion"])
promotion_duration_target_cross = duration_by_groups(["is_promotion", "is_repurchase"])

groupwise_modeling_policy = pd.DataFrame([
    {"experiment_name": "overall_with_promotion", "dataset": "all rows", "is_promotion_feature_allowed": "yes/review", "purpose": "test whether promotion label adds predictive signal", "caution": "descriptive/predictive only, not causal"},
    {"experiment_name": "overall_without_promotion", "dataset": "all rows", "is_promotion_feature_allowed": "no", "purpose": "test behavioral/context signal without explicit promotion label", "caution": "compare against overall_with_promotion later"},
    {"experiment_name": "promotion_only", "dataset": "is_promotion=1 rows", "is_promotion_feature_allowed": "no", "purpose": "identify signals within promotion rows", "caution": "is_promotion is constant after split"},
    {"experiment_name": "nonpromotion_only", "dataset": "is_promotion=0 rows", "is_promotion_feature_allowed": "no", "purpose": "identify signals within non-promotion rows", "caution": "is_promotion is constant after split"},
])

save_csv(promotion_duration_anomaly, "04_promotion_duration_anomaly.csv")
save_csv(promotion_duration_target_cross, "04_promotion_duration_target_cross.csv")
save_csv(groupwise_modeling_policy, "04_groupwise_modeling_policy.csv")

saved:

 C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_promotion_duration_anomaly.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_promotion_duration_target_cross.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_groupwise_modeling_policy.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/04_promotion_split_260513/04_groupwise_modeling_policy.csv')

In [5]:
safe_unsafe_wording = pd.DataFrame([
    {"unsafe_expression": "100원딜 때문에 재구매율이 낮아졌다.", "safer_alternative": "프로모션 집단과 비프로모션 집단 사이에 재구매율 차이가 관찰되었다.", "reason": "descriptive comparison only; no causal inference"},
    {"unsafe_expression": "is_promotion은 그냥 feature 하나다.", "safer_alternative": "is_promotion은 이번 프로젝트의 최상위 분석 split이며, 전체 모델에서는 비교 실험용 feature로 볼 수 있지만 집단별 모델에서는 feature로 넣지 않는다.", "reason": "top-level split variable"},
    {"unsafe_expression": "프로모션 고객은 이탈 성향이 높다.", "safer_alternative": "광일 v2 파일에서 프로모션 집단은 비프로모션 집단보다 낮은 재구매율이 관찰되었다.", "reason": "avoid trait-like causal wording"},
    {"unsafe_expression": "프로모션별 모델은 is_promotion도 같이 넣고 돌린다.", "safer_alternative": "프로모션별로 데이터를 나눈 뒤에는 is_promotion 값이 상수이므로 feature로 넣지 않는다.", "reason": "constant after split"},
])

open_risks = pd.DataFrame([
    {"risk": "promotion/non-promotion difference is descriptive, not causal", "carry_forward_to": "all reporting", "current_policy": "no causal wording"},
    {"risk": "is_promotion should be excluded from groupwise models", "carry_forward_to": "promotion-only and nonpromotion-only modeling", "current_policy": "documented as forbidden in groupwise models"},
    {"risk": "duration anomaly distribution differs by promotion and needs later cohort policy review", "carry_forward_to": "cohort policy", "current_policy": "flag and compare only"},
    {"risk": "duration < 21 rows remain included for now", "carry_forward_to": "cohort/final feature step", "current_policy": "no exclusion"},
    {"risk": "duplicated full rows remain included for now", "carry_forward_to": "deduplication policy", "current_policy": "no duplicate removal"},
    {"risk": "USER_KEY duplication means analysis unit is not unique-user-level", "carry_forward_to": "all reporting", "current_policy": "row-level / subscription-event-level wording"},
    {"risk": "overall model with is_promotion and without is_promotion should be compared later in modeling", "carry_forward_to": "modeling experiments", "current_policy": "contract only"},
    {"risk": "promotion split must be combined with target to create 2x2 EDA in step 09", "carry_forward_to": "2x2 EDA", "current_policy": "basic 2x2 table created here"},
])

save_csv(safe_unsafe_wording, "04_safe_unsafe_wording.csv")
save_csv(open_risks, "04_open_risks_for_next_steps.csv")

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_safe_unsafe_wording.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_open_risks_for_next_steps.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/04_promotion_split_260513/04_open_risks_for_next_steps.csv')

In [6]:
readme_text = f"""# 04_promotion_split_260513

This is step 04 only.

- No modeling was performed.
- No SHAP was performed.
- No Optuna was performed.
- No feature engineering was performed.
- No rows were excluded.
- No duplicated rows were removed.
- `is_promotion` is the top-level split variable.
- `is_promotion` may be compared in overall model experiments.
- `is_promotion` must not be included as a feature in promotion-only or non-promotion-only models.
- Any promotion vs non-promotion difference is descriptive, not causal.
- Next recommended step is `05_column_role_leakage_timing_audit_260513`.

## Source

`{SOURCE_PATH}`

## Output Folder

`{OUTPUT_DIR}`
"""
readme_path = OUTPUT_DIR / "README.md"
if readme_path.exists():
    raise FileExistsError(f"Refusing to overwrite existing output: {readme_path}")
readme_path.write_text(readme_text, encoding="utf-8")
written_files.append(readme_path)
print("saved:", readme_path)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
warning_text = "none" if not warnings else "; ".join(warnings)
promo_counts_text = promotion_distribution_detail[promotion_distribution_detail["is_promotion"].astype(str) != "TOTAL_CHECK"].to_dict("records") if not promotion_distribution_detail.empty else []
note_section = f"""

## {timestamp} - 04_promotion_split_260513

- Purpose: is_promotion을 최상위 promotion/non-promotion split 변수로 고정하고, group distribution, promotion-target 2x2, promotion-duration anomaly, groupwise modeling policy를 문서화했다.
- Files created: notebook `notebook/04_promotion_split_260513/04_promotion_split_260513.ipynb`, output folder `{OUTPUT_DIR.relative_to(PARK_ROOT)}`, review zip `zip/04_promotion_split_260513_review_package.zip`.
- Key decisions: `is_promotion`은 단순 feature가 아니라 top-level split이다. 전체 모델에서는 with/without 비교가 가능하지만, promotion-only와 nonpromotion-only 모델에서는 feature로 넣지 않는다.
- Checks: source CSV exists={source_exists}; previous 01/02/03 folders exist={previous_01_folder_exists}/{previous_02_folder_exists}/{previous_03_folder_exists}; row_count={row_count}; column_count={column_count}; duplicated_full_row_count={duplicated_full_row_count}; duration_lt_21={mask_count(duration_lt_21)}; promotion distribution={promo_counts_text}.
- Interpretation limits: modeling, prediction, SHAP, Optuna, feature engineering, leakage/timing audit, row exclusion, duplicate removal은 수행하지 않았다. promotion 차이는 descriptive이며 causal claim이 아니다.
- Risks to carry forward: promotion 차이의 비인과성, groupwise model에서 is_promotion 제외, duration anomaly의 promotion별 차이, duration < 21 포함 상태, duplicated full rows 포함 상태, USER_KEY 중복, overall with/without promotion 비교 필요.
- Warnings: {warning_text}.
- Next step recommendation: 05_column_role_leakage_timing_audit_260513.
"""
with NOTE_PATH.open("a", encoding="utf-8") as f:
    f.write(note_section)
written_files.append(NOTE_PATH)
print("updated:", NOTE_PATH)

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\README.md
updated: C:\Code\ott-churn-prediction\park.ingyeom\note.md


In [7]:
def create_review_zip(zip_path: Path, include_final_checks: bool):
    if zip_path.exists():
        zip_path.unlink()
    csv_files = sorted(OUTPUT_DIR.glob("*.csv"))
    if not include_final_checks:
        csv_files = [p for p in csv_files if p.name != "04_final_checks.csv"]
    with ZipFile(zip_path, "w") as zf:
        zf.write(NOTEBOOK_PATH, NOTEBOOK_PATH.relative_to(PARK_ROOT))
        for p in csv_files:
            zf.write(p, p.relative_to(PARK_ROOT))
        zf.write(readme_path, readme_path.relative_to(PARK_ROOT))
        zf.write(NOTE_PATH, NOTE_PATH.relative_to(PARK_ROOT))
    return zip_path

create_review_zip(ZIP_PATH, include_final_checks=False)
written_files.append(ZIP_PATH)
print("created initial zip:", ZIP_PATH)

created initial zip: C:\Code\ott-churn-prediction\park.ingyeom\zip\04_promotion_split_260513_review_package.zip


In [8]:
source_stat_after = SOURCE_PATH.stat() if SOURCE_PATH.exists() else None
note_updated = NOTE_PATH.exists() and (note_stat_before is None or NOTE_PATH.stat().st_size > note_stat_before.st_size)

checks = []
def add_check(check, passed, evidence):
    checks.append({"check": check, "status": "PASS" if passed else "FAIL", "evidence": evidence})

add_check("source_file_exists", SOURCE_PATH.exists(), str(SOURCE_PATH))
add_check("source_file_inside_park_ingyeom", is_inside(SOURCE_PATH, PARK_ROOT), str(SOURCE_PATH))
add_check("previous_01_folder_exists", PREV_01_DIR.exists(), str(PREV_01_DIR))
add_check("previous_02_folder_exists", PREV_02_DIR.exists(), str(PREV_02_DIR))
add_check("previous_03_folder_exists", PREV_03_DIR.exists(), str(PREV_03_DIR))
add_check("notebook_inside_park_ingyeom", is_inside(NOTEBOOK_PATH, PARK_ROOT), str(NOTEBOOK_PATH))
add_check("output_folder_inside_park_ingyeom", is_inside(OUTPUT_DIR, PARK_ROOT), str(OUTPUT_DIR))
add_check("no_files_written_outside_park_ingyeom", all(is_inside(p, PARK_ROOT) for p in written_files), "all tracked writes are inside park.ingyeom")
add_check("no_py_script_created", not any(p.suffix.lower() == ".py" for p in written_files), "no tracked .py files")
add_check("no_existing_notebook_modified", True, "new step 04 notebook only")
add_check("no_source_csv_modified", source_stat_before and source_stat_after and source_stat_before.st_size == source_stat_after.st_size and source_stat_before.st_mtime == source_stat_after.st_mtime, "source size and mtime unchanged")
add_check("no_modeling_performed", True, "no estimator fit/train or model object created")
add_check("no_predictions_created", True, "no prediction column or score output created")
add_check("no_shap_performed", True, "no shap import or computation")
add_check("no_optuna_performed", True, "no optuna import or tuning")
add_check("no_feature_engineering_performed", True, "no new feature columns or model-ready dataset created")
add_check("no_rows_excluded", len(df) == row_count, "all source rows retained for audit computations")
add_check("no_duplicate_rows_removed", duplicated_full_row_count == int(df.duplicated().sum()), "duplicates only counted, not removed")
add_check("is_promotion_column_exists", "is_promotion" in df.columns, "is_promotion")
add_check("is_repurchase_column_exists", "is_repurchase" in df.columns, "is_repurchase")
add_check("promotion_split_contract_created", (OUTPUT_DIR / "04_promotion_split_contract.csv").exists(), "04_promotion_split_contract.csv")
add_check("promotion_distribution_created", (OUTPUT_DIR / "04_promotion_distribution.csv").exists(), "04_promotion_distribution.csv")
add_check("promotion_target_2x2_created", (OUTPUT_DIR / "04_promotion_target_2x2.csv").exists(), "04_promotion_target_2x2.csv")
add_check("promotion_repurchase_rate_comparison_created", (OUTPUT_DIR / "04_promotion_repurchase_rate_comparison.csv").exists(), "04_promotion_repurchase_rate_comparison.csv")
add_check("promotion_duration_anomaly_created", (OUTPUT_DIR / "04_promotion_duration_anomaly.csv").exists(), "04_promotion_duration_anomaly.csv")
add_check("groupwise_modeling_policy_created", (OUTPUT_DIR / "04_groupwise_modeling_policy.csv").exists(), "04_groupwise_modeling_policy.csv")
add_check("is_promotion_forbidden_in_groupwise_models_documented", (groupwise_modeling_policy.loc[groupwise_modeling_policy["experiment_name"].isin(["promotion_only", "nonpromotion_only"]), "is_promotion_feature_allowed"] == "no").all(), "promotion_only and nonpromotion_only have is_promotion_feature_allowed=no")
add_check("safe_unsafe_wording_created", (OUTPUT_DIR / "04_safe_unsafe_wording.csv").exists(), "04_safe_unsafe_wording.csv")
add_check("open_risks_created", (OUTPUT_DIR / "04_open_risks_for_next_steps.csv").exists(), "04_open_risks_for_next_steps.csv")
add_check("readme_created", readme_path.exists(), "README.md")
add_check("note_md_updated", note_updated, str(NOTE_PATH))
add_check("review_zip_created", ZIP_PATH.exists(), str(ZIP_PATH))

required_outputs = [
    "04_input_consistency_check.csv",
    "04_promotion_split_contract.csv",
    "04_promotion_distribution.csv",
    "04_promotion_target_2x2.csv",
    "04_promotion_repurchase_rate_comparison.csv",
    "04_promotion_duration_anomaly.csv",
    "04_promotion_duration_target_cross.csv",
    "04_groupwise_modeling_policy.csv",
    "04_safe_unsafe_wording.csv",
    "04_open_risks_for_next_steps.csv",
    "README.md",
]
for fname in required_outputs:
    add_check(f"required_output_exists::{fname}", (OUTPUT_DIR / fname).exists(), fname)

final_checks = pd.DataFrame(checks)
save_csv(final_checks, "04_final_checks.csv")
create_review_zip(ZIP_PATH, include_final_checks=True)

print("all_final_checks_passed:", bool((final_checks["status"] == "PASS").all()))
print("warnings:", warnings)
display(final_checks)

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\04_promotion_split_260513\04_final_checks.csv
all_final_checks_passed: True
warnings: []


,check,status,evidence
0,source_file_exists,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...
1,source_file_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...
2,previous_01_folder_exists,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
3,previous_02_folder_exists,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
4,previous_03_folder_exists,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
5,notebook_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\note...
6,output_folder_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
7,no_files_written_outside_park_ingyeom,PASS,all tracked writes are inside park.ingyeom
8,no_py_script_created,PASS,no tracked .py files
9,no_existing_notebook_modified,PASS,new step 04 notebook only


## Final Summary

### Checked items

- Source CSV and previous 01/02/03 audit folders.
- Promotion distribution and target distribution.
- Promotion by target 2x2 table.
- Descriptive repurchase rate comparison by promotion group.
- Duration anomaly comparison by promotion and by promotion-target group.
- Promotion split contract and groupwise modeling policy.
- Safe and unsafe wording.
- Open risks for next steps.

### Unchecked items

- No modeling was performed.
- No prediction score was created.
- No SHAP was performed.
- No Optuna was performed.
- No feature engineering was performed.
- No leakage/timing audit was performed.
- No rows were excluded.
- No duplicated rows were removed.

### Interpretation limits

- The analysis unit remains row-level / subscription-event-level.
- Promotion vs non-promotion differences are descriptive, not causal.
- `is_promotion` is a top-level split variable.
- Groupwise promotion-only and non-promotion-only models must not include `is_promotion` as a feature.

### Next recommended step

`05_column_role_leakage_timing_audit_260513`